# Advanced Problems with Solutions: Random Samples in Python

This notebook contains advanced practice problems about `random.sample`, sampling without replacement, reproducible sampling, comparison with `random.choices`, and practical card-deck simulations.

## Learning goals

By the end, you should be able to:

- Explain the difference between sampling with replacement and without replacement.
- Use `random.sample` safely and correctly.
- Understand why `sample` cannot choose more items than the population contains.
- Use fixed seeds for reproducible samples.
- Detect duplicate selections using `Counter`.
- Build and sample from a deck of cards.
- Use full-population sampling as a non-mutating shuffle.
- Design robust helper functions around random sampling.

In [1]:
import random
from collections import Counter
from itertools import combinations

## Problem 1: Sampling Without Replacement

`random.choices(population, k=n)` samples with replacement, while `random.sample(population, k=n)` samples without replacement.

### Task

1. Create a population `['a', 'b', 'c']`.
2. Use `choices` to draw 10 items.
3. Use `sample` to draw 3 items.
4. Prove that the `sample` result contains no duplicates.
5. Explain why `sample(population, 10)` is impossible for this population.

### Solution

In [2]:
population = ['a', 'b', 'c']

rng = random.Random(0)

with_replacement = rng.choices(population, k=10)
without_replacement = rng.sample(population, k=3)

print("choices result:", with_replacement)
print("sample result:", without_replacement)

assert len(without_replacement) == len(set(without_replacement))

try:
    rng.sample(population, k=10)
except ValueError as ex:
    print("Expected error:", ex)

choices result: ['c', 'c', 'b', 'a', 'b', 'b', 'c', 'a', 'b', 'b']
sample result: ['a', 'c', 'b']
Expected error: Sample larger than population or is negative


### Explanation

`sample` removes each chosen item from future eligibility. Therefore, it cannot return more elements than exist in the population. `choices`, however, samples with replacement, so duplicates are allowed and `k` can be larger than the population size.

## Problem 2: Safe Sampling Helper

Write a function `safe_sample(population, k, seed=None)` that wraps `random.sample` with clearer validation.

### Requirements

The function should:

1. Reject negative `k`.
2. Reject `k > len(population)`.
3. Return a reproducible sample when a seed is supplied.
4. Not mutate the original population.

### Solution

In [3]:
def safe_sample(population, k, seed=None):
    if k < 0:
        raise ValueError("k cannot be negative")
    
    if k > len(population):
        raise ValueError("k cannot be larger than the population size")
    
    rng = random.Random(seed)
    return rng.sample(population, k)


items = list(range(10))
original = items.copy()

sample_1 = safe_sample(items, 5, seed=123)
sample_2 = safe_sample(items, 5, seed=123)

print(sample_1)

assert sample_1 == sample_2
assert items == original
assert len(sample_1) == 5
assert len(sample_1) == len(set(sample_1))

for bad_k in [-1, 11]:
    try:
        safe_sample(items, bad_k)
    except ValueError as ex:
        print("Correctly rejected:", ex)

[0, 4, 1, 6, 3]
Correctly rejected: k cannot be negative
Correctly rejected: k cannot be larger than the population size


### Explanation

`random.sample` already raises errors for invalid `k`, but a wrapper can provide clearer messages and local seeding without touching global random state.

## Problem 3: Full Sample as a Non-Mutating Shuffle

Sampling the entire population with `random.sample(population, k=len(population))` returns all items in random order.

### Task

1. Write a function `shuffled_copy(items, seed=None)`.
2. It should return a shuffled version of the input.
3. It must not mutate the original list.
4. It must be reproducible for the same seed.

### Solution

In [4]:
def shuffled_copy(items, seed=None):
    rng = random.Random(seed)
    return rng.sample(items, k=len(items))


items = list(range(20))
original = items.copy()

shuffled_1 = shuffled_copy(items, seed=0)
shuffled_2 = shuffled_copy(items, seed=0)

print(shuffled_1)

assert shuffled_1 == shuffled_2
assert items == original
assert sorted(shuffled_1) == sorted(items)
assert len(shuffled_1) == len(items)
assert len(shuffled_1) == len(set(shuffled_1))

[12, 13, 1, 8, 15, 6, 19, 4, 7, 5, 9, 3, 2, 11, 17, 0, 14, 16, 18, 10]


### Explanation

`random.shuffle` mutates a list in place. `random.sample(items, len(items))` creates a shuffled copy instead.

## Problem 4: Detecting Replacement by Counting Duplicates

You are given two unknown random selection functions. One samples with replacement and the other samples without replacement.

### Task

1. Generate 20 selected cards from a 52-card deck using `choices`.
2. Generate 20 selected cards from the same deck using `sample`.
3. Use `Counter` to detect duplicates.
4. Identify which result came from replacement.

### Solution

In [5]:
def build_deck():
    suits = ('C', 'D', 'H', 'S')
    ranks = tuple(range(2, 11)) + tuple('JQKA')
    return [f"{rank}{suit}" for suit in suits for rank in ranks]


def duplicate_report(cards):
    counts = Counter(cards)
    return {card: count for card, count in counts.items() if count > 1}


deck = build_deck()
rng = random.Random(42)

selection_with_replacement = rng.choices(deck, k=20)
selection_without_replacement = rng.sample(deck, k=20)

duplicates_with_replacement = duplicate_report(selection_with_replacement)
duplicates_without_replacement = duplicate_report(selection_without_replacement)

print("With replacement:", selection_with_replacement)
print("Duplicates:", duplicates_with_replacement)
print()
print("Without replacement:", selection_without_replacement)
print("Duplicates:", duplicates_without_replacement)

assert duplicates_without_replacement == {}
assert len(selection_without_replacement) == len(set(selection_without_replacement))

With replacement: ['9H', '3C', '3D', 'KC', 'AH', 'JH', '9S', '6C', '10D', '3C', 'KC', '2H', '3C', 'QC', '9H', '4H', 'KC', '6H', '5S', '2C']
Duplicates: {'9H': 2, '3C': 3, 'KC': 3}

Without replacement: ['AS', 'QC', '7S', '3H', '10D', '6D', 'JC', '2D', '10S', '8C', '7C', 'KD', '5S', 'JD', 'AH', '5D', '4C', '5H', '9C', '3S']
Duplicates: {}


### Explanation

A `sample` from a deck cannot repeat the same physical card. `choices` can repeat cards because each draw is made from the full deck again.

## Problem 5: Build and Validate a Deck

Build a standard 52-card deck.

### Task

1. Use four suits: clubs, diamonds, hearts, spades.
2. Use thirteen ranks: 2 through 10, J, Q, K, A.
3. Confirm that the deck has 52 cards.
4. Confirm that every card is unique.
5. Deal a reproducible 5-card hand without replacement.

### Solution

In [6]:
SUITS = ('C', 'D', 'H', 'S')
RANKS = tuple(range(2, 11)) + tuple('JQKA')

deck = [f"{rank}{suit}" for suit in SUITS for rank in RANKS]

assert len(deck) == 52
assert len(deck) == len(set(deck))

rng = random.Random(0)
hand = rng.sample(deck, k=5)

print("Deck size:", len(deck))
print("Hand:", hand)

assert len(hand) == 5
assert len(hand) == len(set(hand))

Deck size: 52
Hand: ['KD', 'JS', '2H', '4C', '5D']


### Explanation

A deck is a Cartesian product of suits and ranks. Since each card is unique, `sample(deck, 5)` is the correct way to deal a poker-style hand.

## Problem 6: Deal Multiple Poker Hands Correctly

A game deals four players five cards each from the same deck.

### Task

Write `deal_hands(num_players, cards_per_player, seed=None)` that:

1. Samples all required cards without replacement.
2. Splits the sampled cards into hands.
3. Raises an error if the deal requires more than 52 cards.
4. Proves no card appears in more than one hand.

### Solution

In [7]:
def deal_hands(num_players, cards_per_player, seed=None):
    if num_players < 0 or cards_per_player < 0:
        raise ValueError("num_players and cards_per_player must be non-negative")
    
    deck = build_deck()
    total_cards = num_players * cards_per_player
    
    if total_cards > len(deck):
        raise ValueError("Cannot deal more cards than the deck contains")
    
    rng = random.Random(seed)
    dealt_cards = rng.sample(deck, k=total_cards)
    
    return [
        dealt_cards[i * cards_per_player : (i + 1) * cards_per_player]
        for i in range(num_players)
    ]


hands = deal_hands(4, 5, seed=99)

for i, hand in enumerate(hands, start=1):
    print(f"Player {i}:", hand)

all_cards = [card for hand in hands for card in hand]

assert len(all_cards) == 20
assert len(all_cards) == len(set(all_cards))

try:
    deal_hands(11, 5, seed=0)
except ValueError as ex:
    print("Expected error:", ex)

Player 1: ['AD', 'KD', 'AC', 'AH', 'KC']
Player 2: ['3D', '4D', '10C', '7C', '5D']
Player 3: ['KS', '9H', '10H', '6S', '7H']
Player 4: ['QS', '2H', '2D', 'QD', '4S']
Expected error: Cannot deal more cards than the deck contains


### Explanation

The safest approach is to sample all dealt cards at once, then partition them into hands. This guarantees uniqueness across all players.

## Problem 7: Compare `sample` and Repeated `choice`

A student tries to sample without replacement by repeatedly calling `random.choice`.

### Task

1. Write a flawed function that chooses `k` items using repeated `choice`.
2. Demonstrate that duplicates can appear.
3. Rewrite it correctly using `sample`.
4. Explain the difference.

### Solution

In [8]:
def flawed_sample(population, k, seed=None):
    rng = random.Random(seed)
    return [rng.choice(population) for _ in range(k)]


def correct_sample(population, k, seed=None):
    rng = random.Random(seed)
    return rng.sample(population, k)


population = list(range(5))

bad = flawed_sample(population, 10, seed=1)
good = correct_sample(population, 5, seed=1)

print("Flawed repeated choice:", bad)
print("Correct sample:", good)

assert len(bad) != len(set(bad))
assert len(good) == len(set(good))

Flawed repeated choice: [1, 4, 0, 2, 0, 3, 3, 3, 3, 1]
Correct sample: [1, 0, 4, 3, 2]


### Explanation

Repeated `choice` samples with replacement because each call chooses from the full population again. `sample` tracks what has already been chosen and prevents duplicates.

## Problem 8: Random Team Assignment Without Overlap

You have a class of 30 students named `Student 1` through `Student 30`.

### Task

1. Randomly choose 12 students without replacement.
2. Split them into 3 teams of 4.
3. Make the assignment reproducible.
4. Prove that no selected student appears on two teams.

### Solution

In [9]:
def assign_random_teams(students, num_teams, team_size, seed=None):
    total_needed = num_teams * team_size
    
    if total_needed > len(students):
        raise ValueError("Not enough students for the requested teams")
    
    rng = random.Random(seed)
    selected = rng.sample(students, k=total_needed)
    
    return [
        selected[i * team_size : (i + 1) * team_size]
        for i in range(num_teams)
    ]


students = [f"Student {i}" for i in range(1, 31)]
teams = assign_random_teams(students, num_teams=3, team_size=4, seed=2024)

for i, team in enumerate(teams, start=1):
    print(f"Team {i}:", team)

selected_students = [student for team in teams for student in team]

assert len(selected_students) == 12
assert len(selected_students) == len(set(selected_students))
assert teams == assign_random_teams(students, 3, 4, seed=2024)

Team 1: ['Student 16', 'Student 6', 'Student 24', 'Student 19']
Team 2: ['Student 10', 'Student 7', 'Student 28', 'Student 14']
Team 3: ['Student 9', 'Student 18', 'Student 8', 'Student 30']


### Explanation

This is another practical use of sampling without replacement: assigning people, tasks, or resources where duplication would be invalid.

## Problem 9: Sampling and Probability of a Flush

A 5-card poker hand is a flush if all cards have the same suit.

### Task

1. Simulate 100,000 poker hands using `random.sample`.
2. Estimate the probability of a flush.
3. Compare it to the theoretical probability:

`4 * C(13, 5) / C(52, 5)`

where `C(n, k)` is the number of combinations.

### Solution

In [10]:
def n_choose_k(n, k):
    return len(list(combinations(range(n), k)))


def is_flush(hand):
    suits = [card[-1] for card in hand]
    return len(set(suits)) == 1


def estimate_flush_probability(num_hands, seed=None):
    rng = random.Random(seed)
    deck = build_deck()
    flushes = 0
    
    for _ in range(num_hands):
        hand = rng.sample(deck, k=5)
        if is_flush(hand):
            flushes += 1
    
    return flushes / num_hands


empirical = estimate_flush_probability(100_000, seed=0)
theoretical = 4 * n_choose_k(13, 5) / n_choose_k(52, 5)

print("Empirical probability:", empirical)
print("Theoretical probability:", theoretical)
print("Absolute error:", abs(empirical - theoretical))

assert abs(empirical - theoretical) < 0.002

Empirical probability: 0.00219
Theoretical probability: 0.0019807923169267707
Absolute error: 0.00020920768307322937


### Explanation

A poker hand must be sampled without replacement. Using `choices` would allow impossible hands such as the same card appearing twice.

## Problem 10: Avoid an Inefficient Combination Function

The previous problem used `len(list(combinations(...)))`, which is inefficient for large values.

### Task

Write a better `comb(n, k)` function using arithmetic instead of generating all combinations. Then recompute the theoretical flush probability.

### Solution

In [11]:
def comb(n, k):
    if k < 0 or k > n:
        return 0
    
    k = min(k, n - k)
    numerator = 1
    denominator = 1
    
    for i in range(1, k + 1):
        numerator *= n - (k - i)
        denominator *= i
    
    return numerator // denominator


theoretical_flush = 4 * comb(13, 5) / comb(52, 5)

print(theoretical_flush)

assert comb(13, 5) == 1287
assert comb(52, 5) == 2598960
assert round(theoretical_flush, 6) == round(4 * 1287 / 2598960, 6)

0.0019807923169267707


### Explanation

The arithmetic version computes combinations directly and avoids building potentially huge lists in memory.

## Problem 11: Create a Random Lottery Draw

A lottery draws 6 unique numbers from 1 through 49.

### Task

1. Write `lottery_draw(seed=None)`.
2. Return the selected numbers sorted ascending.
3. Ensure the result has exactly 6 unique numbers.
4. Ensure all numbers are between 1 and 49.
5. Make the draw reproducible.

### Solution

In [12]:
def lottery_draw(seed=None):
    rng = random.Random(seed)
    return sorted(rng.sample(range(1, 50), k=6))


draw_1 = lottery_draw(seed=7)
draw_2 = lottery_draw(seed=7)

print(draw_1)

assert draw_1 == draw_2
assert len(draw_1) == 6
assert len(draw_1) == len(set(draw_1))
assert all(1 <= number <= 49 for number in draw_1)
assert draw_1 == sorted(draw_1)

[4, 5, 10, 21, 26, 42]


### Explanation

Lottery numbers are selected without replacement because the same number cannot be drawn twice in the same draw.

## Problem 12: Design a Reproducible Sampler Class

Create a reusable class called `Sampler`.

### Requirements

The class should:

1. Store its own `random.Random` instance.
2. Provide `.sample(population, k)` for sampling without replacement.
3. Provide `.shuffle_copy(population)` using `sample`.
4. Reject invalid `k` values with helpful errors.
5. Be reproducible when initialized with the same seed.

### Solution

In [13]:
class Sampler:
    def __init__(self, seed=None):
        self.rng = random.Random(seed)
    
    def sample(self, population, k):
        if k < 0:
            raise ValueError("k cannot be negative")
        
        if k > len(population):
            raise ValueError("k cannot be larger than the population size")
        
        return self.rng.sample(population, k)
    
    def shuffle_copy(self, population):
        return self.sample(population, len(population))


sampler_1 = Sampler(seed=100)
sampler_2 = Sampler(seed=100)

items = list(range(10))

sample_1 = sampler_1.sample(items, 4)
sample_2 = sampler_2.sample(items, 4)

print(sample_1)
print(sample_2)

assert sample_1 == sample_2
assert len(sample_1) == len(set(sample_1))

shuffled = Sampler(seed=5).shuffle_copy(items)

print(shuffled)

assert sorted(shuffled) == items
assert items == list(range(10))

[2, 7, 8, 6]
[2, 7, 8, 6]
[9, 4, 5, 6, 7, 8, 0, 1, 3, 2]


### Explanation

Using a class with its own generator avoids accidental interference from global random state and makes randomized behavior easier to reproduce.

## Best-practice checklist

When using `random.sample`:

1. Use `sample` when duplicates are not allowed.
2. Use `choices` when sampling with replacement is intended.
3. Never request more sampled items than the population contains.
4. Use `sample(population, len(population))` to create a shuffled copy.
5. Use `random.Random(seed)` for reproducible local randomness.
6. Use `Counter` or `set` to verify uniqueness when needed.
7. For card games, lotteries, team assignments, and resource allocation, sampling without replacement is usually the correct model.
8. Validate `k` before sampling in production-quality code.
9. Avoid mutating caller-owned data unless mutation is explicitly intended.
10. Store seeds when reproducibility matters.